# Tahap 2: Training & Evaluasi Collaborative Filtering

Pipeline utama untuk Evaluasi & Training Implicit Collaborative Filtering:
1. Evaluasi grid search menggunakan ALS (Alternating Least Squares)
2. Evaluasi grid search menggunakan BPR (Bayesian Personalized Ranking)
3. Fit kembali menggunakan metrik tebaik pada keseluruhan data `train + validation`
4. Uji dan tampilkan *test metric* pada `Leave-One-Out Test Setup`.

In [1]:
import json
import pickle
import time
from pathlib import Path
from itertools import product
import pandas as pd

# Load custom components
from data_prep import (
    load_cf_splits,
    get_matrix_dims,
    build_user_item_matrix,
    build_user_history,
    prepare_loo_eval,
)
from evaluator import evaluate_loo, metrics_to_dataframe
from models.als_model import ALSModel
from models.bpr_model import BPRModel

ROOT = Path("../../food.com")
OUTPUT_DIR = Path("outputs")
MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"

for d in [MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_NEGATIVES = 99
K_VALUES = (5, 10, 20)
SEED = 42

## 1. Load Data Pembagian Matrix
Memuat matrix yang sudah digenerate dari `01_build_cf_split.py`. 

In [2]:
print("Memuat Dataset train, test, validation...")
train_df, val_df, test_df = load_cf_splits()
n_users, n_items = get_matrix_dims(train_df, val_df, test_df)

print("Membangun matriks dan riwayat interaksi...")
train_matrix = build_user_item_matrix(train_df, n_users, n_items, binary=True)
print(f"Matrix User-Item: {train_matrix.shape}")

train_history = build_user_history(train_df)
train_val_history = build_user_history(train_df, val_df)

# Ekstrak Evaluasi LOO Negatives
val_loo = prepare_loo_eval(val_df, train_history, n_items, N_NEGATIVES, SEED)
test_loo = prepare_loo_eval(test_df, train_val_history, n_items, N_NEGATIVES, SEED)

print("Kesiapan matrix telah diselesaikan.")

Memuat Dataset train, test, validation...
[data_prep] CF LOO split loaded -- Train: 622,730  Val: 19,123  Test: 19,123
[data_prep] Matrix dims -- Users: 19,123  Items: 75,017
Membangun matriks dan riwayat interaksi...
Matrix User-Item: (19123, 75017)
[data_prep] LOO users for evaluation: 19,123
[data_prep] LOO eval records ready: 19,123
[data_prep] LOO users for evaluation: 19,123
[data_prep] LOO eval records ready: 19,123
Kesiapan matrix telah diselesaikan.


## 2. Menjalankan Parameter Grid Search (Validation Phase)
Pada contoh tutorial `jupytext` ini, kita menggunakan setup hiperparameter ringkas supaya cepat berjalan. 

In [3]:
def run_grid_search(ModelClass, param_grid, train_matrix, val_loo, model_name):
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))
    
    print(f"\n[{model_name}] Memulai Grid Search...")
    best_hr10, best_params, best_model = -1.0, None, None
    all_results = []
    
    for combo in combos:
        params = dict(zip(keys, combo))
        print(f"Menguji: {params}")
        
        # Training
        model = ModelClass(**params)
        model.fit(train_matrix)
        
        # Validation
        metrics = evaluate_loo(model, val_loo, k_values=K_VALUES, verbose=False)
        all_results.append({**params, **metrics})
        
        if metrics["HR@10"] > best_hr10:
            best_hr10 = metrics["HR@10"]
            best_params = params
            best_model = model
            
    print(f"--> [Terbaik {model_name}] Validation HR@10: {best_hr10:.4f} @ {best_params}\n")
    return best_model, best_params, best_hr10, pd.DataFrame(all_results)

# Grid hyperparameters ringkas
ALS_GRID_QUICK = {
    "factors": [32, 64],
    "regularization": [0.01],
    "iterations": [20],
    "alpha": [20.0, 40.0],
}
BPR_GRID_QUICK = {
    "factors": [32, 64],
    "learning_rate": [0.01, 0.05],
    "regularization": [0.01],
    "iterations": [100],
}

_, best_als_params, als_val_hr10, als_res = run_grid_search(ALSModel, ALS_GRID_QUICK, train_matrix, val_loo, "ALS")
_, best_bpr_params, bpr_val_hr10, bpr_res = run_grid_search(BPRModel, BPR_GRID_QUICK, train_matrix, val_loo, "BPR")


[ALS] Memulai Grid Search...
Menguji: {'factors': 32, 'regularization': 0.01, 'iterations': 20, 'alpha': 20.0}


c:\Users\aemer\miniconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

[ALS] Trained — factors=32, reg=0.01, iter=20, alpha=20.0
Menguji: {'factors': 32, 'regularization': 0.01, 'iterations': 20, 'alpha': 40.0}


  0%|          | 0/20 [00:00<?, ?it/s]

...

[BPR] Trained — factors=32, lr=0.05, reg=0.01, iter=100
Menguji: {'factors': 64, 'learning_rate': 0.01, 'regularization': 0.01, 'iterations': 100}


  0%|          | 0/100 [00:00<?, ?it/s]

[BPR] Trained — factors=64, lr=0.01, reg=0.01, iter=100
Menguji: {'factors': 64, 'learning_rate': 0.05, 'regularization': 0.01, 'iterations': 100}


  0%|          | 0/100 [00:00<?, ?it/s]

[BPR] Trained — factors=64, lr=0.05, reg=0.01, iter=100
--> [Terbaik BPR] Validation HR@10: 0.3662 @ {'factors': 32, 'learning_rate': 0.01, 'regularization': 0.01, 'iterations': 100}



## 3. Final Retraining & Testing
Mengajarkan `best metrics parameters` kembali pada gabungan data `train+val`, kemudian diuji di Test set.

In [4]:
print("Melakukan Re-training model pada Train+Validation...")
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
train_val_matrix = build_user_item_matrix(train_val_df, n_users, n_items, binary=True)

final_als = ALSModel(**best_als_params)
final_als.fit(train_val_matrix)

final_bpr = BPRModel(**best_bpr_params)
final_bpr.fit(train_val_matrix)

# Mendapatkan hasil evaluasi:
print("\n[EVALUASI TEST SET - ALS]")
als_test = evaluate_loo(final_als, test_loo, k_values=K_VALUES, verbose=True)

print("\n[EVALUASI TEST SET - BPR]")
bpr_test = evaluate_loo(final_bpr, test_loo, k_values=K_VALUES, verbose=True)

Melakukan Re-training model pada Train+Validation...


  0%|          | 0/20 [00:00<?, ?it/s]

[ALS] Trained — factors=32, reg=0.01, iter=20, alpha=40.0


  0%|          | 0/100 [00:00<?, ?it/s]

[BPR] Trained — factors=32, lr=0.01, reg=0.01, iter=100

[EVALUASI TEST SET - ALS]
  Evaluating… 1,912/19,123 (10%)
  Evaluating… 3,824/19,123 (20%)
  Evaluating… 5,736/19,123 (30%)
  Evaluating… 7,648/19,123 (40%)
  Evaluating… 9,560/19,123 (50%)
  Evaluating… 11,472/19,123 (60%)
  Evaluating… 13,384/19,123 (70%)
  Evaluating… 15,296/19,123 (80%)
  Evaluating… 17,208/19,123 (90%)
  Evaluating… 19,120/19,123 (100%)

[EVALUASI TEST SET - BPR]
  Evaluating… 1,912/19,123 (10%)
  Evaluating… 3,824/19,123 (20%)
  Evaluating… 5,736/19,123 (30%)
  Evaluating… 7,648/19,123 (40%)
  Evaluating… 9,560/19,123 (50%)
  Evaluating… 11,472/19,123 (60%)
  Evaluating… 13,384/19,123 (70%)
  Evaluating… 15,296/19,123 (80%)
  Evaluating… 17,208/19,123 (90%)
  Evaluating… 19,120/19,123 (100%)


## 4. Menyimpan Model ke Log Repositori

In [5]:
all_test = {"ALS": als_test, "BPR": bpr_test}
results_df = metrics_to_dataframe(all_test)

print("\nKESIMPULAN PERBANDINGAN MODEL:")
print(results_df.to_string(float_format=lambda x: f"{x:.4f}"))

results_df.to_csv(RESULT_DIR / "final_results.csv")

# Tentukan model the best of the best
best_name = "BPR" if bpr_test["HR@10"] >= als_test["HR@10"] else "ALS"
best_final = final_bpr if best_name == "BPR" else final_als

with open(MODEL_DIR / "best_cf_model.pkl", "wb") as f:
    pickle.dump({"name": best_name, "model": best_final}, f)

print(f"\nExport Artifact ({best_name}) Berhasil -> output folder.")


KESIMPULAN PERBANDINGAN MODEL:
        HR@5  HR@10  HR@20    MRR
Model                            
ALS   0.3849 0.4654 0.5602 0.2951
BPR   0.2542 0.3634 0.5059 0.1901

Export Artifact (ALS) Berhasil -> output folder.
